[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_08/4_ai.ipynb)

# Day 8: AI and ML (easy)

**How to work:** Answer concept questions out loud in 60 to 90 seconds before opening the solution: definition, why it matters, one example, one trade-off.

**Today (26 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Many trees, two philosophies | easy | 5 |
| Q2 | Score a split by hand | easy | 5 |
| Q3 | Does the fancy model win on churn? | easy | 7 |
| Q4 | The noise column that looks important | easy | 6 |
| Q5 | Tame an overfitting booster (review) | easy | 3 |

**Handbook:** [AI and ML](https://mahsa-agz.github.io/ds-handbook/#ai), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

- `telco`: IBM Telco churn, 7,043 customers, one row = one customer. `Churn` (Yes/No), `tenure` (months), `Contract`, `MonthlyCharges`, `TotalCharges` (text with blanks), 9 service columns, demographics.
- `titanic`: 891 passengers, one row = one passenger. `survived` (0/1), `pclass`, `sex`, `age` (has NaN), `sibsp`, `parch`, `fare`, `embarked`, plus derived columns (`alive`, `who`, ...).

The setup cell loads every table into a pandas DataFrame (from your Drive folder, or downloads it). Everything runs on the free Colab CPU in seconds. Numbers in the solutions come from scikit-learn 1.8; other versions can differ in the last digit.

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import io, zipfile, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.width", 120)

SEABORN = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/"
TELCO_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
BANK_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

def load_telco():
    """IBM Telco churn, 7,043 customers. Target: Churn (Yes/No)."""
    return pd.read_csv(data_path("telco_churn.csv", TELCO_URL))

def load_seaborn(name):
    """titanic, penguins, diamonds, mpg, tips ... from the seaborn-data repo."""
    return pd.read_csv(data_path(name + ".csv", SEABORN + name + ".csv"))

def load_bank():
    """UCI Bank Marketing (bank-additional-full), 41,188 calls. Target: y (yes/no). The source is a zip in a zip."""
    path = data_path("bank_marketing.csv")
    if not os.path.exists(path):
        print("downloading bank_marketing (zip)")
        outer = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(BANK_URL).read()))
        inner = zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip")))
        with open(path, "wb") as f:
            f.write(inner.read("bank-additional/bank-additional-full.csv"))
    return pd.read_csv(path, sep=";")

def check(fn, cases, tol=1e-6):
    """Runs fn on each (args, expected) case; numbers and arrays are compared with a tolerance."""
    ok = 0
    for args, want in cases:
        args = args if isinstance(args, tuple) else (args,)
        try:
            got = fn(*args)
            good = got is not None and np.shape(got) == np.shape(want) and bool(
                np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), atol=tol))
        except Exception as e:
            got, good = "error: " + repr(e)[:120], False
        ok += good
        shown = " ".join(repr(got).split())
        shown = shown if len(shown) < 100 else shown[:97] + "..."
        print(("PASS " if good else "FAIL ") + f"{fn.__name__}(...) -> {shown}" + ("" if good else f"   expected {want!r}"))
    print(f"{ok}/{len(cases)} passed")

telco = load_telco()
print("telco", telco.shape)
titanic = load_seaborn("titanic")
print("titanic", titanic.shape)

---
### Q1. Many trees, two philosophies

*easy, about 5 min*

In about 90 seconds: how does a decision tree choose a split, and how do random forests and gradient boosting combine trees differently? Say which error (bias or variance) each one mainly reduces and name the two or three hyperparameters you would tune first for each.

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: 'random forest versus XGBoost'. Pattern: bagging (parallel, averages) versus boosting (sequential, corrects residuals).

</details>

<details><summary><b>Hint 2</b></summary>

1. Split = the feature and threshold with the largest impurity drop (Gini/entropy, or variance for regression). 2. Forest: deep trees on bootstrap samples with random feature subsets, averaged. 3. Boosting: shallow trees, each fitted to the gradient of the loss (residuals), added with a small learning rate.

</details>

<details><summary><b>Solution</b></summary>

**Model answer (about 90 s):**

"A tree greedily tries every feature and threshold and keeps the split that most reduces impurity: Gini `1 - sum p_k^2` or entropy for classification, variance for regression. It repeats in each child until a stopping rule. Single deep trees have low bias but very high variance.

A **random forest** grows many deep trees in parallel, each on a bootstrap sample and with a random subset of features at each split, and averages them. Averaging decorrelated trees mainly cuts **variance**. It is hard to break: tune `max_features`, `min_samples_leaf`, and use enough trees (more trees never overfit, they just cost time).

**Gradient boosting** grows shallow trees one after another; each new tree fits the negative gradient of the loss (for squared error, the residuals) of the current ensemble, and is added with a small learning rate. It mainly cuts **bias**, and it can overfit if you add too many trees. Tune `learning_rate` together with the number of trees (use early stopping on a validation set), `max_depth` or number of leaves, and subsampling of rows and columns. XGBoost, LightGBM and CatBoost add regularization, histogram splits and categorical handling.

On tabular data, tuned boosting usually wins; a forest is a strong, low-effort baseline."

**Why:** The bagging-versus-boosting contrast (parallel and variance versus sequential and bias) and the 'learning rate times number of trees' coupling are what examiners listen for.

**Common mistakes:** Saying random forests reduce bias. Saying more trees always overfit a forest (they do not; they can in boosting). Forgetting that boosting fits gradients, not just 'the misclassified points' (that is AdaBoost's reweighting view).

**Learn more:** [ai-trees-boosting](https://mahsa-agz.github.io/ds-handbook/#ai-trees-boosting)

</details>

---
### Q2. Score a split by hand

*easy, about 5 min*

Write `gini(labels)` (Gini impurity `1 - sum p_k^2`, 0 for an empty list) and `split_gain(left, right)`: the parent's Gini minus the size-weighted Gini of the two children.

Example: parent `[0, 0, 0, 1, 1, 1, 1, 0]` split into `[0, 0, 0, 1]` and `[1, 1, 1, 0]`: the parent has Gini 0.5, each child 0.375, so the gain is 0.125.

Before running: what is the gain of a split that leaves both children with the same class mix as the parent?

In [ ]:
def gini(labels):
    # your code here
    pass

def split_gain(left, right):
    # your code here
    pass

# tests
check(gini, [([0, 0, 1, 1], 0.5), ([1, 1, 1], 0.0), ([0, 1, 2], 2 / 3), ([0, 0, 0, 1], 0.375)])
check(split_gain, [(([0, 0], [1, 1]), 0.5),
                   (([0, 0, 0, 1], [1, 1, 1, 0]), 0.125),
                   (([0, 1], [0, 1]), 0.0),
                   (([0, 0, 0, 0, 1], [1]), 4 / 9 - 5 / 6 * 0.32)])

<details><summary><b>Hint 1</b></summary>

Signal: 'how does a tree pick a split'. Pattern: impurity decrease (Gini).

</details>

<details><summary><b>Hint 2</b></summary>

1. Class shares p_k with `np.unique(..., return_counts=True)`. 2. `1 - sum(p**2)`. 3. Gain = `gini(parent) - (n_L / n) gini(L) - (n_R / n) gini(R)`.

</details>

<details><summary><b>Solution</b></summary>

```python
def gini(labels):
    labels = np.asarray(labels)
    if len(labels) == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return float(1 - np.sum(p ** 2))

def split_gain(left, right):
    """Impurity of the parent minus the size-weighted impurity of the two children."""
    parent = np.concatenate([left, right])
    n = len(parent)
    return gini(parent) - (len(left) / n * gini(left) + len(right) / n * gini(right))
```

**Why:** Gini is the chance that two random draws from the node have different classes. A split that keeps the same mix in both children has gain 0, so the tree will not prefer it. The last test shows why trees like peeling off small pure groups: isolating one positive from `[0, 0, 0, 0, 1, 1]` gives a gain of 0.178, while with only a few rows such splits are often noise; `min_samples_leaf` guards against that.

**Complexity:** `gini` is `O(n)` (`O(n log n)` with `np.unique`). A real tree sorts each feature once and scans all thresholds, `O(n log n)` per feature per node.

**Common mistakes:** Forgetting to weight the children by size. Using the counts instead of the proportions.

**Learn more:** [ai-trees-boosting](https://mahsa-agz.github.io/ds-handbook/#ai-trees-boosting)

</details>

---
### Q3. Does the fancy model win on churn?

*easy, about 7 min*

Use `telco` with the same preparation as day 5 (drop `customerID`, numeric `TotalCharges` with blanks as 0, scale the 4 numeric columns, one-hot the rest, 75/25 stratified split with `random_state=0`).

Compare the test ROC AUC of:

- `LogisticRegression(max_iter=1000)`
- `RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0)`
- `HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=3, random_state=0)`

Which wins? What would you tell a manager who says "just use XGBoost, it always wins"?

In [ ]:
# telco is loaded. Your code here

<details><summary><b>Hint 1</b></summary>

Signal: model comparison on a small tabular dataset. Pattern: bagging versus boosting versus a linear baseline, all on the same split.

</details>

<details><summary><b>Hint 2</b></summary>

1. One ColumnTransformer reused in three pipelines (`sparse_output=False` so every model gets a dense array). 2. Loop over a dict of models; fit, then `roc_auc_score` on `predict_proba(X_te)[:, 1]`.

</details>

<details><summary><b>Solution</b></summary>

```python
import time
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = telco.drop(columns="customerID").copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
y = (df.pop("Churn") == "Yes").astype(int)
num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = [c for c in df.columns if c not in num]
X_tr, X_te, y_tr, y_te = train_test_split(df, y, test_size=0.25, stratify=y, random_state=0)
prep = ColumnTransformer([("num", StandardScaler(), num),
                          ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])
models = {
    "logistic regression": LogisticRegression(max_iter=1000),
    "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "gradient boosting": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=3,
                                                        random_state=0),
}
for name, m in models.items():
    pipe = make_pipeline(prep, m).fit(X_tr, y_tr)
    print(f"{name:20s} test AUC {roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1]):.3f}")
# out: logistic regression  test AUC 0.844
# out: random forest        test AUC 0.842
# out: gradient boosting    test AUC 0.844
```

**Why:** All three land at about the same test AUC: 0.844 for logistic regression and boosting, 0.842 for the forest. A 0.002 gap on 1,761 test rows is noise. Telco churn is driven by a few strong, mostly monotonic signals (contract, tenure, charges, fibre internet), which a linear model on one-hot features already captures. For the manager: boosting often wins on large tabular data with interactions, but not always; here the logistic model is equally good, faster, calibrated and explainable, so it is the better default. Decide with cross-validation and the business metric, not by reputation.

**Complexity:** The forest builds 300 trees on about 5,300 rows (a second or two with `n_jobs=-1`); histogram boosting with 200 shallow trees is under a second.

**Common mistakes:** Comparing models on different splits or different preprocessing. Declaring a winner from a 0.005 AUC gap on one split of 1,761 rows. Tuning the boosted model a lot and the baseline not at all.

**Learn more:** [ai-trees-boosting](https://mahsa-agz.github.io/ds-handbook/#ai-trees-boosting), [ai-classification-metrics](https://mahsa-agz.github.io/ds-handbook/#ai-classification-metrics)

</details>

---
### Q4. The noise column that looks important

*easy, about 6 min*

Use `titanic`. Features: `pclass`, `female` (0/1), `age` (missing as -1), `fare`, `sibsp`, `parch`, and `random_id`, a column of pure noise (a random permutation of 0..890, seed 0). Split 70/30 (stratified, `random_state=0`) and fit `RandomForestClassifier(n_estimators=300, random_state=0)`.

Print, for each feature, the forest's built-in (impurity) importance and the permutation importance on the **test** set (`n_repeats=10, random_state=0`), sorted by impurity importance.

Where does `random_id` rank under each method, and which method would you show a stakeholder?

In [ ]:
# titanic is loaded. Your code here

<details><summary><b>Hint 1</b></summary>

Signal: 'which features matter?' from a tree ensemble. Pattern: impurity importance is biased toward high-cardinality features; permutation importance on held-out data is not.

</details>

<details><summary><b>Hint 2</b></summary>

1. Build the feature frame and the noise column. 2. `rf.feature_importances_`. 3. `sklearn.inspection.permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=0)`.

</details>

<details><summary><b>Solution</b></summary>

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

d = titanic[["survived", "pclass", "sex", "age", "fare", "sibsp", "parch"]].copy()
d["female"] = (d.pop("sex") == "female").astype(int)
d["age"] = d["age"].fillna(-1)                      # trees can split missing (-1) apart
d["random_id"] = np.random.default_rng(0).permutation(len(d))   # pure noise with many unique values
y = d.pop("survived")
X_tr, X_te, y_tr, y_te = train_test_split(d, y, test_size=0.3, stratify=y, random_state=0)

rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
perm = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=0)
print(f"{'feature':10s} impurity  permutation(test)")
for name, imp, pm in sorted(zip(d.columns, rf.feature_importances_, perm.importances_mean), key=lambda t: -t[1]):
    print(f"{name:10s} {imp:8.3f}  {pm:8.3f}")
# out: feature    impurity  permutation(test)
# out: female        0.241     0.205
# out: fare          0.231     0.033
# out: random_id     0.196    -0.004
# out: age           0.163     0.046
# out: pclass        0.082     0.041
# out: sibsp         0.046    -0.001
# out: parch         0.042     0.016
```

**Why:** The forest's impurity importance ranks pure noise third (0.196), above age and class. Impurity importance is measured on the training data, and a feature with 891 unique values offers many split points, so deep trees use it to memorise individual passengers. Permutation importance on the test set asks 'how much does test accuracy drop if I shuffle this column?', and `random_id` gets -0.004, i.e. nothing. It also shows that `fare` (0.231 versus 0.033) is inflated for the same reason. Show the stakeholder permutation importance on held-out data (or SHAP values), and say that it measures predictive use, not causal effect.

**Complexity:** 300 trees on 623 rows plus 7 features x 10 shuffles of test predictions: a second or two.

**Common mistakes:** Reporting impurity importance as 'what drives survival'. Computing permutation importance on the training set (it then also rewards memorised noise). Forgetting that correlated features share or hide importance under both methods (for example `pclass` and `fare`).

**Learn more:** [ai-trees-boosting](https://mahsa-agz.github.io/ds-handbook/#ai-trees-boosting), [ai-features-imbalance](https://mahsa-agz.github.io/ds-handbook/#ai-features-imbalance)

</details>

---
### Q5. Tame an overfitting booster (review)

*easy, about 3 min*

Your LightGBM model has validation AUC 0.81 and training AUC 0.99, and validation AUC starts falling after about 150 trees. Which knobs do you turn, in what order, and how do you choose their values without fooling yourself?

*Your answer (double-click to write):*

<details><summary><b>Hint 1</b></summary>

Signal: boosting plus a train/validation gap. Pattern: regularization of boosting and honest tuning with cross-validation.

</details>

<details><summary><b>Hint 2</b></summary>

Learning rate with early stopping; tree size; row/column subsampling; L1/L2 on leaf values; minimum data per leaf. Tune with CV on training data only, then one test evaluation.

</details>

<details><summary><b>Solution</b></summary>

**Model answer:** "First, early stopping: keep the number of trees where validation loss is best (here about 150) rather than a fixed large number. Then lower the learning rate (say 0.1 to 0.03) and let early stopping pick more trees; smaller steps generalise better. Next, smaller trees: fewer leaves or lower depth, and a higher minimum number of samples per leaf. Then randomness: row subsampling (bagging fraction 0.7 to 0.9) and column subsampling. Finally explicit L1/L2 penalties on leaf weights.

To choose values honestly: tune on the training data with K-fold CV (random or Bayesian search over a few of these), use the early-stopping set inside each fold, and keep the test set untouched for one final number. If the train/validation gap stays large, check for leakage or a train/validation distribution shift before tuning more."

**Why:** It is the boosting version of day 6: regularization is anything that limits complexity, and its strength must be chosen on validation data, not on the test set.

**Common mistakes:** Adding more trees. Using the test set for early stopping. Tuning ten knobs at once on one split.

**Learn more:** [ai-regularization-cv](https://mahsa-agz.github.io/ds-handbook/#ai-regularization-cv), [ai-trees-boosting](https://mahsa-agz.github.io/ds-handbook/#ai-trees-boosting)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Done for today.** Finish with the [theory flashcards for day 8](https://mahsa-agz.github.io/ds-handbook/#day-8).